# AGM5733 - Métodos Observacionais em Climatologia e Meteorologia de Mesoescala 

Aula: Filtros Temporais

## Bibliotecas

In [ ]:
import io
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "font.size": 11,
})

NAVY, VERM, VERDE, LARANJA, CINZA = "#1b3a6b", "#c1272d", "#2a7f62", "#e08214", "#8a8f98"


try:
    from ipywidgets import interact, IntSlider, FloatSlider
    WIDGETS = True

    def sl(valor, minimo, maximo, passo, rotulo):
        """Controle deslizante que só recalcula quando você solta o cursor."""
        Cls = IntSlider if isinstance(passo, int) and isinstance(valor, int) else FloatSlider
        return Cls(value=valor, min=minimo, max=maximo, step=passo,
                   description=rotulo, continuous_update=False,
                   style={"description_width": "initial"})
except ImportError:
    WIDGETS = False
    print("ipywidgets indisponível: gráficos estáticos, com os valores padrão")

### Dados

Índice **AO** (*Arctic Oscillation*) diário do CPC/NOAA, o modo anular do Hemisfério Norte, calculado a partir da altura geopotencial em 1000 hPa, de 1950 até hoje, ~28 000 dias, e é adimensional (desvio normalizado).

In [ ]:
URL_BASE = "https://ftp.cpc.ncep.noaa.gov/cwlinks/"
REPO_DADOS = ("https://raw.githubusercontent.com/dariohhossoda/"
              "agm5733-filtrostemporais/main/dados/")
ARQUIVO = "norm.daily.ao.cdas.z1000.19500101_current.csv"


def localizar_dado(arquivo, url_original=None, timeout=30):
    """Procura o arquivo: cópia local -> repositório da aula -> fonte original."""
    tentativas = [("cópia local em dados/", os.path.join("dados", arquivo)),
                  ("repositório da aula", REPO_DADOS + arquivo)]
    if url_original:
        tentativas.append(("fonte original", url_original))

    for rotulo, endereco in tentativas:
        try:
            if endereco.startswith("http"):
                with urllib.request.urlopen(endereco, timeout=timeout) as resposta:
                    return rotulo, io.BytesIO(resposta.read())
            if os.path.exists(endereco):
                return rotulo, endereco
        except Exception:
            continue
    raise FileNotFoundError(arquivo)


def carregar_indice(arquivo=ARQUIVO):
    """Índice diário do CPC/NOAA, indexado por data."""
    fonte, alvo = localizar_dado(arquivo, URL_BASE + arquivo)
    df = pd.read_csv(alvo)
    df.columns = ["ano", "mes", "dia", "valor"]
    data = pd.to_datetime(
        df[["ano", "mes", "dia"]].rename(columns={"ano": "year", "mes": "month", "dia": "day"})
    )
    serie = pd.Series(df["valor"].astype(float).values, index=data,
                      name=arquivo.split(".")[2].upper())
    serie.attrs["fonte"] = fonte
    return serie


def serie_sintetica(n=28000, semente=0):
    """Plano B: ruído vermelho + sinal intrassazonal + sinal interanual."""
    rng = np.random.default_rng(semente)
    t = np.arange(n)
    ruido = np.zeros(n)
    for i in range(1, n):
        ruido[i] = 0.75 * ruido[i - 1] + rng.normal(0, 1)
    x = (ruido / ruido.std()
         + 0.8 * np.sin(2 * np.pi * t / 45)      # intrassazonal ~45 dias
         + 0.5 * np.sin(2 * np.pi * t / 1500))   # interanual ~4 anos
    idx = pd.date_range("1950-01-01", periods=n, freq="D")
    return pd.Series(x, index=idx, name="SINTETICA")


try:
    ao = carregar_indice()
    print("índice AO, dado real | fonte:", ao.attrs["fonte"])
except Exception as e:
    print("nenhuma fonte respondeu (%s) -> série sintética" % type(e).__name__)
    ao = serie_sintetica()

print(ao.name, "| n =", len(ao), "| de", ao.index.min().date(), "a", ao.index.max().date())
print("valores faltantes:", int(ao.isna().sum()))
ao.head()

In [ ]:
print("datas com NaN:")
print(ao[ao.isna()])

# `ao` guarda o NaN de propósito; `ao_ok` é a versão interpolada, para comparação.
ao_ok = ao.interpolate(limit_direction="both")
x = ao_ok.values
t = ao_ok.index
dt = 1.0 # intervalo de amostragem: 1 dia

In [ ]:
def fig_serie_bruta(ano_ini=1950, anos=80):
    """A série bruta, numa janela de anos à sua escolha."""
    sel = (t.year >= ano_ini) & (t.year < ano_ini + anos)
    if sel.sum() == 0:
        print("janela vazia — a série vai de %d a %d" % (t.year.min(), t.year.max()))
        return
    fig, ax = plt.subplots(figsize=(12, 3.4))
    ax.plot(t[sel], x[sel], color=CINZA, lw=0.4 if sel.sum() > 5000 else 0.9)
    ax.set_title("Índice %s diário — série bruta (%d–%d)"
                 % (ao.name, t[sel].year.min(), t[sel].year.max()))
    ax.set_ylabel("desvio normalizado")
    plt.show()


# if WIDGETS:
#     interact(fig_serie_bruta,
#              ano_ini=sl(1950, int(t.year.min()), int(t.year.max()), 1, "ano inicial"),
#              anos=sl(80, 1, 80, 1, "anos na tela"))
# else:
#     fig_serie_bruta()
fig_serie_bruta()

---
## Análise prévia

In [ ]:
def remover_media_e_tendencia(serie):
    """Remove a média e a reta de mínimos quadrados, devolvendo o resíduo.

    Sempre antes de filtrar: tendência linear não é periódica, então a FFT a
    representa espalhando potência por todas as frequências (vazamento
    espectral), contaminando justamente a banda que se quer isolar.
    """
    y = np.asarray(serie, dtype=float)
    tt = np.arange(len(y), dtype=float)
    d_t = tt - tt.mean()
    d_y = y - y.mean()
    b_ang = (d_t * d_y).sum() / (d_t ** 2).sum()
    a_lin = y.mean() - b_ang * tt.mean()
    return y - (a_lin + b_ang * tt)


def espectro(serie, dt=1.0, remover_tendencia=True, janela=True):
    """Periodograma simples. Devolve (frequências, densidade espectral), sem f=0."""
    s = np.asarray(serie, dtype=float)
    s = s[np.isfinite(s)]
    if remover_tendencia:
        s = remover_media_e_tendencia(s)
    n = len(s)
    w = np.hanning(n) if janela else np.ones(n)
    H = np.fft.rfft(s * w)
    f = np.fft.rfftfreq(n, d=dt)
    pot = (np.abs(H) ** 2) / n
    return f[1:], pot[1:]


def suavizar(p, k=201):
    """Média móvel no espectro, só para enxergar a forma (senão vira grama)."""
    return np.convolve(p, np.ones(k) / k, mode="same")


f, P = espectro(x, dt)


def fig_espectro_bruto(suavizacao=201, janela_hanning=True):
    """Espectro do índice bruto. `suavizacao` só afeta a leitura, não o cálculo."""
    ff, PP = espectro(x, dt, janela=janela_hanning)
    fig, ax = plt.subplots()
    ax.loglog(1 / ff, suavizar(PP, suavizacao), color=NAVY, lw=1.2)
    for periodo, rotulo in [(7, "sinótica\n~7 d"), (45, "intrassazonal\n~45 d"),
                            (365, "anual"), (365 * 4, "interanual")]:
        ax.axvline(periodo, color=CINZA, ls="--", lw=0.8)
        ax.text(periodo, ax.get_ylim()[1] * 0.5, rotulo, rotation=90, fontsize=8,
                color=CINZA, ha="right", va="top")
    ax.set_xlabel("período (dias)"); ax.set_ylabel("densidade espectral")
    ax.set_title("Espectro do índice bruto — suavização de %d pontos" % suavizacao)
    plt.show()


# if WIDGETS:
#     interact(fig_espectro_bruto,
#              suavizacao=sl(201, 1, 601, 20, "suavização"),
#              janela_hanning=True)
# else:
#     fig_espectro_bruto()
fig_espectro_bruto()

---
## Média móvel

$$Y_t = \sum_{k=-m}^{m} W_k X_{t+k}, \qquad W_k = \frac{1}{L}$$

$X_t$ é a série original e $Y_t$ a filtrada; $W_k$ é a **função peso**, com $k$ indo de $-m$ a
$+m$ ($2m+1$ pesos no total); $L$ é o comprimento da média móvel.

O filtro só preserva a fase se os pesos forem **simétricos em torno de $t$**.
A média móvel padrão do Excel e o `rolling()` do pandas sem `center=True` usa apenas pontos passados e atribui o resultado ao último ponto, defasando o sinal em $(L-1)/2$.

In [ ]:
def demo_media_movel(L=31, periodo=60, ruido=0.9):
    """Média móvel centrada × não centrada, numa onda pura enterrada em ruído."""
    rng = np.random.default_rng(7)
    tt = np.arange(400)
    onda = 2 * np.sin(2 * np.pi * tt / periodo)
    teste = pd.Series(onda + rng.normal(0, ruido, tt.size))

    centrada = teste.rolling(L, center=True).mean()   # simétrica  -> sem defasagem
    atrasada = teste.rolling(L).mean()                # assimétrica -> defasada
    defasagem = (L - 1) // 2

    # fração da amplitude da onda que resta após o filtro, isto é, R(f)
    f_onda = 1.0 / periodo
    R = np.sin(np.pi * f_onda * L) / (L * np.sin(np.pi * f_onda))

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(tt, teste, color=CINZA, lw=0.6, label="série (onda + ruído)")
    ax.plot(tt, onda, color="k", lw=1.0, ls=":", label=f"sinal verdadeiro (T = {periodo} d)")
    ax.plot(tt, centrada, color=NAVY, lw=2,
            label="centrada: simétrica, sem defasagem")
    ax.plot(tt, atrasada, color=VERM, lw=2,
            label=f"não centrada: atrasada {defasagem} dias")
    ax.axhline(0, color="k", lw=0.7)
    ax.set_xlabel("tempo (dias)"); ax.set_ylabel("amplitude")
    ax.set_title("L = %d | atraso da versão não centrada = %d dias | "
                 "amplitude residual da onda: R = %+.2f" % (L, defasagem, R))
    ax.legend(fontsize=9, ncol=2, loc="upper right")
    fig.tight_layout()
    plt.show()

In [ ]:
if WIDGETS:
    interact(demo_media_movel,
             L=sl(31, 3, 101, 2, "L (ímpar)"),
             periodo=sl(60, 5, 120, 5, "período (d)"),
             ruido=sl(0.9, 0.0, 3.0, 0.1, "ruído"))
else:
    demo_media_movel()

---
## $R(f)$ e a oscilação de Gibbs

$R(f)$ é a **resposta de frequência**: o fator pelo qual o filtro multiplica a amplitude da onda
de frequência $f$. Para pesos simétricos ela é real:

$$R(f) = W_0 + 2\sum_{k=1}^{m} W_k \cos(2\pi f k \Delta t)$$

Para a média móvel de comprimento $L$, aplicada $p$ vezes, há forma fechada:

$$R_p(f) = \left[\frac{\operatorname{sen}(\pi f L \Delta t)}{L \operatorname{sen}(\pi f \Delta t)}\right]^{p}$$

com $p$ = número de aplicações sucessivas do mesmo filtro. Para o triangular 1-2-1, também
aplicado $p$ vezes:

$$R(f) = \left[\cos^{2}(\pi f \Delta t)\right]^{p}$$

In [ ]:
def resposta_frequencia(pesos, freqs, dt=1.0):
    """R(f) de um conjunto simétrico de pesos ordenados de k=-m a k=+m."""
    pesos = np.asarray(pesos, dtype=float)
    m = (len(pesos) - 1) // 2
    k = np.arange(1, m + 1)
    arg = 2.0 * np.pi * np.outer(freqs, k) * dt
    return pesos[m] + 2.0 * (pesos[m + 1:] * np.cos(arg)).sum(axis=1)


def fr_media_movel(freqs, L, p=1, dt=1.0):
    f = np.asarray(freqs, dtype=float)
    den = L * np.sin(np.pi * f * dt)
    r = np.where(np.isclose(den, 0.0), 1.0, np.sin(np.pi * f * L * dt) / np.where(den == 0, 1.0, den))
    return r ** p


def fr_121(freqs, p=1, dt=1.0):
    return (np.cos(np.pi * np.asarray(freqs) * dt) ** 2) ** p


# conferência: a fórmula fechada bate com a soma dos cossenos?
freqs = np.linspace(1e-9, 0.5, 3000)
w_ma3 = np.full(3, 1 / 3)
print("máx |numérico - analítico| (média móvel L=3):",
      np.max(np.abs(resposta_frequencia(w_ma3, freqs) - fr_media_movel(freqs, 3))))

$R(f)$ da média móvel (esquerda) e do 1-2-1 (direita).

In [ ]:
def fig_resposta_ma_121(L=3, p=1, L_comparacao=11):
    """R(f) da média móvel (esquerda) e do 1-2-1 (direita), com p aplicações."""
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    r = fr_media_movel(freqs, L)
    ax[0].axhline(0, color="k", lw=0.8)
    ax[0].plot(freqs, r, color=NAVY, label=f"média móvel L={L}, p=1")
    if p > 1:
        ax[0].plot(freqs, fr_media_movel(freqs, L, p=p), color=NAVY, ls=":",
                   label=f"L={L}, p={p}")
    ax[0].plot(freqs, fr_media_movel(freqs, L_comparacao), color=VERM,
               label=f"média móvel L={L_comparacao}")
    ax[0].fill_between(freqs, 0, r, where=r < 0, color=VERM, alpha=0.2)
    ax[0].set_title("Média móvel: R(f) < 0 corresponde à oscilação de Gibbs")
    ax[0].legend(fontsize=9)

    for pp, cor in zip([1, p, 5, 20], [NAVY, VERDE, LARANJA, VERM]):
        ax[1].plot(freqs, fr_121(freqs, pp), color=cor, label=f"1-2-1, p={pp}")
    ax[1].axhline(0, color="k", lw=0.8)
    ax[1].set_title("Filtro 1-2-1: R(f) ≥ 0 em toda a faixa, sem Gibbs")
    ax[1].legend(fontsize=9)

    for a in ax:
        a.set_xlabel("frequência (ciclos/dia)"); a.set_ylabel("R(f)"); a.set_xlim(0, 0.5)
    plt.tight_layout(); plt.show()


if WIDGETS:
    interact(fig_resposta_ma_121,
             L=sl(3, 3, 51, 2, "L (ímpar)"),
             p=sl(1, 1, 8, 1, "passagens p"),
             L_comparacao=sl(11, 3, 51, 2, "L de comparação"))
else:
    fig_resposta_ma_121()

In [ ]:
def aplicar_media_movel(serie, L, p=1):
    """Aplica p vezes a média móvel de comprimento L. Bordas não confiáveis."""
    y = np.asarray(serie, dtype=float)
    w = np.full(L, 1.0 / L)
    for _ in range(p):
        y = np.convolve(y, w, mode="same")
    return y


def demo_gibbs(L=3, p=1, periodo=2.5):
    """R(f) da média móvel, e o efeito do filtro numa onda pura desse período."""
    f_onda = 1.0 / periodo
    R = float(fr_media_movel(np.array([f_onda]), L, p)[0])

    tt = np.arange(400, dtype=float)
    onda = np.cos(2 * np.pi * f_onda * tt)
    filtrada = aplicar_media_movel(onda, L, p)

    # ~8 ciclos no meio da série: com muitos ciclos na tela a inversão de
    # fase, que é o ponto aqui, fica ilegível
    largura = int(np.clip(round(8 * periodo), 30, 200))
    ini = 200 - largura // 2
    meio = slice(ini, ini + largura)

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))

    ff = np.linspace(1e-9, 0.5, 3000)
    Rf = fr_media_movel(ff, L, p)
    ax[0].axhline(0, color="k", lw=0.8)
    ax[0].plot(ff, Rf, color=NAVY, lw=1.6)
    ax[0].fill_between(ff, 0, Rf, where=Rf < 0, color=VERM, alpha=0.25,
                       label="R(f) < 0 (Gibbs)")
    ax[0].axvline(f_onda, color=VERDE, lw=1.2, ls="--")
    ax[0].plot([f_onda], [R], "o", color=VERDE, ms=8)
    ax[0].set_xlim(0, 0.5)
    ax[0].set_ylim(-0.45, 1.05)
    ax[0].set_xlabel("frequência (ciclos/dia)")
    ax[0].set_ylabel("R(f)")
    ax[0].legend(fontsize=8, loc="upper right")
    ax[0].set_title("L = %d, p = %d | R = %+.3f em f = %.3f" % (L, p, R, f_onda))

    # um marcador por dia: em período de 2,5 d a onda tem 2 ou 3 pontos por ciclo
    ax[1].plot(tt[meio], onda[meio], "o-", color=CINZA, lw=1.0, ms=3,
               label="onda de entrada")
    ax[1].plot(tt[meio], filtrada[meio], "o-", color=(VERDE if R >= 0 else VERM),
               lw=2.0, ms=3.5, label="depois do filtro")
    ax[1].axhline(0, color="k", lw=0.8)
    ax[1].set_ylim(-1.2, 1.55)
    ax[1].set_xlabel("tempo (dias)")
    ax[1].set_ylabel("amplitude")
    ax[1].legend(fontsize=8, loc="upper right")

    if R < -0.02:
        diag = "R < 0: amplitude x %.2f, com inversão de fase" % abs(R)
    elif R < 0.02:
        diag = "R = 0: onda eliminada"
    else:
        diag = "R > 0: amplitude x %.2f, fase preservada" % R
    ax[1].set_title(diag)

    fig.tight_layout()
    plt.show()

In [ ]:
if WIDGETS:
    interact(demo_gibbs,
             L=sl(3, 3, 25, 2, "L (ímpar)"),
             p=sl(1, 1, 5, 1, "passagens p"),
             periodo=sl(2.5, 2.0, 20.0, 0.1, "período (d)"))
else:
    demo_gibbs()

---
## Filtro de Lanczos

$$W_0 = 2 f_c \Delta, \qquad
W_k = \underbrace{\frac{\operatorname{sen}(2\pi f_c k \Delta)}{\pi k}}_{\text{sinc (filtro ideal)}}
\cdot
\underbrace{\frac{\operatorname{sen}(\pi k/n)}{\pi k/n}}_{\sigma_k \text{ (fator de Lanczos)}},
\quad k = \pm 1, \dots, \pm n$$

O sinc é o filtro ideal. Truncá-lo em $k = \pm n$ é o que **cria** Gibbs; o fator $\sigma_k$
leva os pesos suavemente a zero na borda e reduz o Gibbs a quase nada.

$f_c$ é a **frequência de corte**, definida por convenção onde $R(f_c) = 0{,}5$.

> **$n$ é o número de pesos de cada lado, e não o tamanho da série.**

In [ ]:
def pesos_lanczos_baixa(fc, n, sigma=True, dt=1.0):
    """Pesos do Lanczos passa-baixa: 2n+1 pesos, corte em fc (ciclos por dt)."""
    k = np.arange(-n, n + 1, dtype=float)
    w = np.empty_like(k)
    nz = k != 0
    kk = k[nz]
    w[k == 0] = 2.0 * fc * dt
    ideal = np.sin(2.0 * np.pi * fc * kk * dt) / (np.pi * kk)
    if sigma:
        arg = np.pi * kk / n
        ideal = ideal * np.sin(arg) / arg
    w[nz] = ideal
    return w


def pesos_lanczos_alta(fc, n, sigma=True, dt=1.0):
    """Passa-alta: Z = X - Y  =>  W' = delta - W,  R_alta = 1 - R_baixa."""
    w = -pesos_lanczos_baixa(fc, n, sigma, dt)
    w[n] += 1.0
    return w


def pesos_lanczos_banda(fc1, fc2, n, sigma=True, dt=1.0):
    """Passa-banda: diferença de dois passa-baixa (fc1 < fc2)."""
    assert fc1 < fc2, "fc1 deve ser menor que fc2"
    return pesos_lanczos_baixa(fc2, n, sigma, dt) - pesos_lanczos_baixa(fc1, n, sigma, dt)


def aplicar_pesos(serie, pesos):
    """Convolução simétrica Y_t = sum_k W_k X_{t+k}. Bordas viram NaN — de propósito."""
    s = np.asarray(serie, dtype=float)
    p = np.asarray(pesos, dtype=float)
    m = (len(p) - 1) // 2
    y = np.convolve(s, p[::-1], mode="same")
    y[:m] = np.nan
    if m:
        y[-m:] = np.nan
    return y

In [ ]:
fc, n = 1 / 30, 60
k = np.arange(-n, n + 1)
w_sinc = pesos_lanczos_baixa(fc, n, sigma=False)
w_lanc = pesos_lanczos_baixa(fc, n, sigma=True)

def fig_sinc_vs_lanczos(corte=30, n=60):
    """Pesos e R(f) do sinc truncado e do Lanczos, para corte de `corte` dias."""
    fc_ = 1.0 / corte
    kk = np.arange(-n, n + 1)
    ws = pesos_lanczos_baixa(fc_, n, sigma=False)
    wl = pesos_lanczos_baixa(fc_, n, sigma=True)

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    ax[0].plot(kk, ws, color=CINZA, label="sinc truncado (sem σ)")
    ax[0].plot(kk, wl, color=NAVY, label="Lanczos (com σ)")
    ax[0].set_xlabel("k"); ax[0].set_ylabel("$W_k$")
    ax[0].set_title("Pesos do filtro: corte %d d, n = %d, %d pesos" % (corte, n, 2 * n + 1))
    ax[0].legend(fontsize=9)

    ffr = np.linspace(1e-6, 0.25, 3000)
    ax[1].plot(ffr, resposta_frequencia(ws, ffr), color=CINZA, ls=":", label="sem σ (Gibbs)")
    ax[1].plot(ffr, resposta_frequencia(wl, ffr), color=NAVY, label="com σ")
    ax[1].axvline(fc_, color=VERM, ls="--", lw=1)
    ax[1].axhline(0.5, color=CINZA, lw=0.8, ls="--")
    ax[1].set_xlabel("frequência (ciclos/dia)"); ax[1].set_ylabel("R(f)")
    ax[1].set_title("Resposta de frequência: R(fc) = %.4f (meia-potência)"
                    % resposta_frequencia(wl, np.array([fc_]))[0])
    ax[1].legend(fontsize=9)
    plt.tight_layout(); plt.show()


if WIDGETS:
    interact(fig_sinc_vs_lanczos,
             corte=sl(30, 3, 200, 1, "corte (dias)"),
             n=sl(60, 5, 400, 5, "n (pesos de cada lado)"))
else:
    fig_sinc_vs_lanczos()

### Escolha de $n$

Regra prática: **$n$ precisa ser da ordem do período de corte** em passos de amostragem, isto é, $n \gtrsim 1/(f_c\,\Delta)$. Se $n$ for pequeno demais, o filtro não preserva nem a média da série ($R(0) \ne 1$), ainda que o resultado possa parecer razoável no gráfico.

In [ ]:
fc_ano = 1 / 365


def fig_escolha_de_n(corte=365, n=200):
    """R(f) do passa-baixa para o n escolhido, contra as referências de sempre."""
    fc_ = 1.0 / corte
    ffr = np.linspace(1e-6, 4 / corte, 3000)
    fig, ax = plt.subplots()
    for nn, cor in zip([corte // 4, corte // 2, corte, 2 * corte],
                       [VERM, LARANJA, VERDE, NAVY]):
        w = pesos_lanczos_baixa(fc_, max(nn, 2))
        r0 = resposta_frequencia(w, np.array([0.0]))[0]
        ax.plot(ffr, resposta_frequencia(w, ffr), color=cor, lw=1.0, alpha=0.45,
                label=f"n={max(nn, 2)}  →  R(0)={r0:.2f}")
    w = pesos_lanczos_baixa(fc_, n)
    r0 = resposta_frequencia(w, np.array([0.0]))[0]
    ax.plot(ffr, resposta_frequencia(w, ffr), color="k", lw=2.2,
            label=f"n={n} (escolhido)  →  R(0)={r0:.3f}")
    ax.axhline(1, color="k", lw=0.8, ls="--")
    ax.axvline(fc_, color=CINZA, ls="--", lw=1)
    ax.set_xlabel("frequência (ciclos/dia)"); ax.set_ylabel("R(f)")
    ax.set_title("Corte em %d dias: com n = %d, R(0) = %.3f%s"
                 % (corte, n, r0, "" if abs(r0 - 1) < 0.02 else " (n insuficiente: R(0) difere de 1)"))
    ax.legend(fontsize=8)
    plt.show()


if WIDGETS:
    interact(fig_escolha_de_n,
             corte=sl(365, 20, 1000, 5, "corte (dias)"),
             n=sl(200, 10, 1500, 10, "n"))
else:
    fig_escolha_de_n()

---
## Passa-banda 20–100 dias (intrassazonal / MJO)

**MJO** = *Madden–Julian Oscillation*, Oscilação de Madden–Julian: par convecção–circulação que se propaga para leste ao longo dos trópicos, com ciclo de 30–60 dias. É o principal modo de variabilidade **intrassazonal**, e a razão de a banda de 20–100 dias existir.

$$W_k^{\text{banda}} = W_k(f_{c2}) - W_k(f_{c1}), \qquad f_{c1} = 1/100,\; f_{c2} = 1/20 \text{ ciclos/dia}$$

In [ ]:
fc1, fc2, n_b = 1 / 100, 1 / 20, 100
w_banda = pesos_lanczos_banda(fc1, fc2, n_b)

def fig_resposta_banda(periodo_curto=20, periodo_longo=100, n=100):
    """R(f) do passa-banda de Lanczos entre os dois períodos de corte."""
    if periodo_curto >= periodo_longo:
        print("o período curto tem de ser menor que o longo")
        return
    f_c1, f_c2 = 1.0 / periodo_longo, 1.0 / periodo_curto
    w = pesos_lanczos_banda(f_c1, f_c2, n)
    ffr = np.linspace(1e-6, min(0.5, 4 * f_c2), 4000)
    fig, ax = plt.subplots()
    ax.plot(ffr, resposta_frequencia(w, ffr), color=NAVY,
            label="Lanczos banda, %d pesos" % (2 * n + 1))
    ax.axhline(0, color="k", lw=0.7)
    ax.axvspan(f_c1, f_c2, color=VERDE, alpha=0.12)
    ax.text(np.sqrt(f_c1 * f_c2), 0.15, "%d–%d dias" % (periodo_curto, periodo_longo),
            ha="center", color=VERDE)
    ax.set_xlabel("frequência (ciclos/dia)"); ax.set_ylabel("R(f)"); ax.legend()
    ax.set_title("Passa-banda %d–%d d com n = %d" % (periodo_curto, periodo_longo, n))
    plt.show()


if WIDGETS:
    interact(fig_resposta_banda,
             periodo_curto=sl(20, 2, 200, 1, "período curto (d)"),
             periodo_longo=sl(100, 5, 800, 5, "período longo (d)"),
             n=sl(100, 10, 600, 10, "n"))
else:
    fig_resposta_banda()

In [ ]:
# aplicado à série real (versão sem NaN)
banda = aplicar_pesos(x, w_banda)
w_baixa365 = pesos_lanczos_baixa(1 / 365, 730)
baixa = aplicar_pesos(x, w_baixa365)

print("pontos perdidos nas bordas: banda =", 2 * n_b, "| passa-baixa =", 2 * 730)


def fig_decomposicao(periodo_curto=20, periodo_longo=100, corte_baixa=365):
    """A série bruta e as duas bandas que os filtros separam dela."""
    if periodo_curto >= periodo_longo:
        print("o período curto tem de ser menor que o longo")
        return
    wb = pesos_lanczos_banda(1 / periodo_longo, 1 / periodo_curto, periodo_longo)
    yb = aplicar_pesos(x, wb)
    wl = pesos_lanczos_baixa(1 / corte_baixa, 2 * corte_baixa)
    yl = aplicar_pesos(x, wl)

    fig, ax = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
    ax[0].plot(t, x, color=CINZA, lw=0.4); ax[0].set_ylabel("bruto")
    ax[1].plot(t, yb, color=VERDE, lw=0.4)
    ax[1].set_ylabel("%d–%d d" % (periodo_curto, periodo_longo))
    ax[2].plot(t, yl, color=NAVY, lw=1.0); ax[2].axhline(0, color="k", lw=0.7)
    ax[2].set_ylabel("> %d d" % corte_baixa); ax[2].set_xlabel("ano")
    ax[0].set_title("%s decomposto por banda; pontos perdidos nas bordas: %d e %d"
                    % (ao.name, 2 * periodo_longo, 4 * corte_baixa))
    plt.tight_layout(); plt.show()


if WIDGETS:
    interact(fig_decomposicao,
             periodo_curto=sl(20, 2, 200, 1, "período curto (d)"),
             periodo_longo=sl(100, 5, 800, 5, "período longo (d)"),
             corte_baixa=sl(365, 100, 1500, 5, "corte do passa-baixa (d)"))
else:
    fig_decomposicao()

### Efeito de série com dado faltante

Aplicação do mesmo filtro na série original, com o `NaN` de 2003-04-30 mantido:

In [ ]:
baixa_com_nan = aplicar_pesos(ao.values, w_baixa365)
perdidos = int(np.isnan(baixa_com_nan).sum()) - 2 * 730

def fig_estrago_nan(n=730, corte=365):
    """O mesmo passa-baixa na série interpolada e na série com um NaN no meio."""
    w = pesos_lanczos_baixa(1 / corte, n)
    y_ok = aplicar_pesos(x, w)
    y_nan = aplicar_pesos(ao.values, w)
    apagados = int(np.isnan(y_nan).sum()) - 2 * n

    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.plot(t, y_ok, color=NAVY, lw=1.0, label="série interpolada")
    ax.plot(t, y_nan, color=VERM, lw=1.6, label="série com 1 NaN")
    ax.set_title("Efeito de um NaN em 2003-04-30: %d pontos filtrados perdidos (= 2n+1 = %d)"
                 % (apagados, 2 * n + 1))
    ax.legend(); plt.show()


if WIDGETS:
    interact(fig_estrago_nan,
             n=sl(730, 10, 1200, 10, "n"),
             corte=sl(365, 30, 1000, 5, "corte (dias)"))
else:
    fig_estrago_nan()

---
## Filtro de Murakami

$$Y_t = C\,(X_t - X_{t-2}) + A\,Y_{t-1} + B\,Y_{t-2}$$

$A$, $B$ e $C$ são os coeficientes do filtro, que não se confundem com os $A_k$, $B_k$ de Fourier.
Como usa apenas o passado, o filtro defasa o sinal. A correção é aplicá-lo duas vezes, uma
para frente e outra para trás no tempo. A defasagem da ida cancela a da volta e a resposta efetiva vira $|H(f)|^2$.

Antes de aplicar: **remover média e tendência linear**.

In [ ]:
def murakami_coef(fc1, fc2, dt=1.0):
    """Coeficientes (b, a) do passa-banda recursivo de 2ª ordem.

    Zeros em f = 0 e em Nyquist; polos dados pela frequência central
    sqrt(fc1*fc2) e pela largura fc2 - fc1.
    """
    f0 = np.sqrt(fc1 * fc2)
    alpha = np.tan(np.pi * (fc2 - fc1) * dt)
    w0 = 2 * np.pi * f0 * dt
    C = alpha / (1 + alpha)
    b = np.array([C, 0.0, -C])
    a = np.array([1.0, -2 * np.cos(w0) / (1 + alpha), (1 - alpha) / (1 + alpha)])
    return b, a


def fr_recursivo(b, a, freqs, dt=1.0, duas_passadas=True):
    """|H(f)| (ou |H(f)|^2, se aplicado ida-e-volta)."""
    z = np.exp(-1j * 2 * np.pi * np.asarray(freqs) * dt)
    h = np.abs((b[0] + b[1] * z + b[2] * z**2) / (a[0] + a[1] * z + a[2] * z**2))
    return h**2 if duas_passadas else h


b, a = murakami_coef(fc1, fc2)
print("b =", b.round(5))
print("a =", a.round(5))

def fig_lanczos_vs_recursivo(periodo_curto=20, periodo_longo=100, n=100):
    """R(f) dos dois filtros na mesma banda: pesos truncados × recursão."""
    if periodo_curto >= periodo_longo:
        print("o período curto tem de ser menor que o longo")
        return
    f_c1, f_c2 = 1.0 / periodo_longo, 1.0 / periodo_curto
    w = pesos_lanczos_banda(f_c1, f_c2, n)
    b_, a_ = murakami_coef(f_c1, f_c2)

    ffr = np.linspace(1e-6, min(0.5, 4 * f_c2), 4000)
    fig, ax = plt.subplots()
    ax.plot(ffr, resposta_frequencia(w, ffr), color=NAVY,
            label="Lanczos (%d pesos)" % (2 * n + 1))
    ax.plot(ffr, fr_recursivo(b_, a_, ffr, duas_passadas=False), color=LARANJA, ls=":",
            label="recursivo, 1 passada |H|")
    ax.plot(ffr, fr_recursivo(b_, a_, ffr), color=VERM, label="recursivo, 2 passadas |H|²")
    ax.axvspan(f_c1, f_c2, color=VERDE, alpha=0.12)
    ax.set_xlabel("frequência (ciclos/dia)"); ax.set_ylabel("R(f)"); ax.legend()
    ax.set_title("Não-recursivo e recursivo na banda %d–%d dias"
                 % (periodo_curto, periodo_longo))
    plt.show()


if WIDGETS:
    interact(fig_lanczos_vs_recursivo,
             periodo_curto=sl(20, 2, 200, 1, "período curto (d)"),
             periodo_longo=sl(100, 5, 800, 5, "período longo (d)"),
             n=sl(100, 10, 600, 10, "n do Lanczos"))
else:
    fig_lanczos_vs_recursivo()

In [ ]:
def uma_passada(serie, b, a):
    """Uma passada, do começo para o fim da série:

        a[0] y[i] = b[0] x[i] + b[1] x[i-1] + b[2] x[i-2]
                  - a[1] y[i-1] - a[2] y[i-2]

    Só usa passado (i-1, i-2) => defasa.
    """
    x_ = np.asarray(serie, dtype=float)
    y = np.zeros_like(x_)
    for i in range(len(x_)):
        acc = 0.0
        for j in range(3):
            if i - j >= 0:
                acc += b[j] * x_[i - j]
        for j in range(1, 3):
            if i - j >= 0:
                acc -= a[j] * y[i - j]
        y[i] = acc / a[0]
    return y


def aplicar_recursivo(serie, b, a):
    """Filtra ida e volta: a defasagem da ida cancela a da volta e R(f) = |H(f)|^2."""
    y = uma_passada(serie, b, a)
    y = uma_passada(y[::-1], b, a)[::-1]
    return y


x_detrend = remover_media_e_tendencia(x)
banda_rec = aplicar_recursivo(x_detrend, b, a)

ok = np.isfinite(banda)
r = np.corrcoef(banda[ok], banda_rec[ok])[0, 1]

fatia = slice(18000, 19000)   # ~3 anos, para conseguir enxergar
print("NaN nas bordas: Lanczos =", int(np.isnan(banda).sum()),
      "| recursivo =", int(np.isnan(banda_rec).sum()))


def fig_dois_filtros(ano_ini=1999, anos=3):
    """As duas séries filtradas lado a lado, na janela de anos escolhida."""
    sel = (t.year >= ano_ini) & (t.year < ano_ini + anos)
    if sel.sum() == 0:
        print("janela vazia")
        return
    fig, ax = plt.subplots(figsize=(12, 3.6))
    ax.plot(t[sel], banda[sel], color=NAVY, lw=1.4, label="Lanczos (201 pesos)")
    ax.plot(t[sel], banda_rec[sel], color=VERM, lw=1.2, ls="--",
            label="recursivo (5 coef., ida-e-volta)")
    ax.axhline(0, color="k", lw=0.7)
    ax.set_title("Mesma banda por dois filtros: correlação global r = %.3f, sem defasagem" % r)
    ax.legend(); plt.show()


# if WIDGETS:
#     interact(fig_dois_filtros,
#              ano_ini=sl(1999, int(t.year.min()), int(t.year.max()), 1, "ano inicial"),
#              anos=sl(3, 1, 20, 1, "anos na tela"))
# else:
#     fig_dois_filtros()
fig_dois_filtros()

---
## Filtragem no domínio da frequência

1. tomar $X_t$ sem média e sem tendência linear;
2. **FFT** (*Fast Fourier Transform*, o algoritmo rápido da transformada discreta de Fourier):
   $X_t \to H(f)$;
3. multiplicar por $R(\omega)$ (aqui, um "tijolo" ideal) → $H'(f)$;
4. FFT inversa → $Y_t$.

In [ ]:
def filtrar_fft(serie, f_baixa=None, f_alta=None, dt=1.0):
    """Os quatro passos acima: tira tendência, FFT, mascara a banda, FFT inversa."""
    anom = remover_media_e_tendencia(serie)
    H = np.fft.rfft(anom)
    f = np.fft.rfftfreq(len(anom), d=dt)
    R = np.ones_like(f)
    if f_baixa is not None:
        R[f < f_baixa] = 0.0
    if f_alta is not None:
        R[f > f_alta] = 0.0
    return np.fft.irfft(H * R, n=len(anom))


banda_fft = filtrar_fft(x, f_baixa=fc1, f_alta=fc2, dt=dt)
ok = np.isfinite(banda)
print("correlação (tempo × frequência) =", round(float(np.corrcoef(banda[ok], banda_fft[ok])[0, 1]), 4))

def fig_tempo_vs_frequencia(ano_ini=1999, anos=3):
    """O mesmo trecho filtrado nos dois domínios, sobre a série bruta."""
    sel = (t.year >= ano_ini) & (t.year < ano_ini + anos)
    if sel.sum() == 0:
        print("janela vazia")
        return
    fig, ax = plt.subplots(figsize=(12, 3.6))
    ax.plot(t[sel], x[sel], color=CINZA, lw=0.5, label="bruto")
    ax.plot(t[sel], banda[sel], color=NAVY, lw=1.5, label="domínio do tempo (convolução)")
    ax.plot(t[sel], banda_fft[sel], color=VERM, lw=1.1, ls="--",
            label="domínio da frequência (FFT)")
    ax.set_title("Domínio do tempo e da frequência: mesmo sinal, sem defasagem")
    ax.legend(fontsize=9); plt.show()


# if WIDGETS:
#     interact(fig_tempo_vs_frequencia,
#              ano_ini=sl(1999, int(t.year.min()), int(t.year.max()), 1, "ano inicial"),
#              anos=sl(3, 1, 20, 1, "anos na tela"))
# else:
#     fig_tempo_vs_frequencia()
fig_tempo_vs_frequencia()

In [ ]:
m = (len(w_banda) - 1) // 2


def fig_bordas(n=100, dias=600):
    """O começo da série: onde a convolução se cala e a FFT inventa número."""
    w = pesos_lanczos_banda(fc1, fc2, n)
    y_tempo = aplicar_pesos(x, w)
    mm = (len(w) - 1) // 2
    fig, ax = plt.subplots(figsize=(12, 3.4))
    ax.plot(t[:dias], y_tempo[:dias], color=NAVY, lw=1.5,
            label="tempo: NaN nos primeiros %d pontos" % mm)
    ax.plot(t[:dias], banda_fft[:dias], color=VERM, lw=1.2, ls="--",
            label="FFT: valores nas bordas, não confiáveis")
    ax.axvspan(t[0], t[min(mm, dias - 1)], color=LARANJA, alpha=0.15)
    ax.set_title("Início da série: a FFT supõe periodicidade e introduz erro nas bordas")
    ax.legend(fontsize=9); plt.show()


if WIDGETS:
    interact(fig_bordas,
             n=sl(100, 10, 600, 10, "n do Lanczos"),
             dias=sl(600, 100, 4000, 100, "dias na tela"))
else:
    fig_bordas()

### Verificação do espectro após a filtragem

In [ ]:
f0, P0 = espectro(x, dt)


def fig_espectro_filtrado(periodo_curto=20, periodo_longo=100, suavizacao=201):
    """A conferência obrigatória: o espectro do que saiu do filtro."""
    if periodo_curto >= periodo_longo:
        print("o período curto tem de ser menor que o longo")
        return
    y = filtrar_fft(x, f_baixa=1 / periodo_longo, f_alta=1 / periodo_curto, dt=dt)
    f1, P1 = espectro(y, dt)
    fig, ax = plt.subplots()
    ax.loglog(1 / f0, suavizar(P0, suavizacao), color=CINZA, lw=1.1, label="bruto")
    ax.loglog(1 / f1, suavizar(P1, suavizacao), color=VERDE, lw=1.3,
              label="filtrado %d–%d d" % (periodo_curto, periodo_longo))
    ax.axvspan(periodo_curto, periodo_longo, color=VERDE, alpha=0.12)
    ax.set_xlabel("período (dias)"); ax.set_ylabel("densidade espectral")
    ax.set_xlim(2, 5000); ax.legend()
    ax.set_title("Verificação do espectro: variância remanescente fora da faixa de corte")
    plt.show()


if WIDGETS:
    interact(fig_espectro_filtrado,
             periodo_curto=sl(20, 2, 200, 1, "período curto (d)"),
             periodo_longo=sl(100, 5, 800, 5, "período longo (d)"),
             suavizacao=sl(201, 1, 601, 20, "suavização"))
else:
    fig_espectro_filtrado()

---
## Exercícios CLIVAC

> **1) Low-pass filtering.** Use o programa `filtering.lowpass.pro` para aplicar um filtro
> passa-baixa à série `temp_1000mb_lat37N_lon120W_series_only`. Use **365 dias** como período
> de corte. **(a)** interprete a série filtrada; **(b)** use o programa de espectro de potência
> da aula anterior, calcule o espectro da série filtrada e interprete o resultado.
>
> **2) Band-pass filtering.** Use o programa `filtering.bandpass.pro` na **mesma série**, com
> períodos de corte de **20 e 100 dias**. **(a)** interprete a série filtrada; **(b)** calcule o
> espectro da série filtrada e interprete o resultado.

| Programa IDL | O que é | Função |
|---|---|---|
| `sub.fft.low.filter.pro` | passa-baixa por FFT | `filtersub_low` |
| `sub.fft.filter.pro` | passa-banda por FFT | `filtersub_band` |
| `filtering.bandpass.pro` | lê o `.dat`, filtra e grava | células do Exercício 2 |

### A série do exercício

Temperatura diária em **1000 hPa** no ponto de grade **37°N, 120°W**, no Vale Central da
Califórnia, de 1979 a 2010, em **kelvin**. É o mesmo arquivo que o cabeçalho de
`filtering.bandpass.pro` documenta, com `mtot = 11680`.

Observações:

- **Unidade física.** Ao contrário do índice AO (normalizado, média zero), aqui a média é
  ~292 K e a variância sai em K². Média e tendência têm de sair antes de qualquer filtro.
- **Calendário de 365 dias.** São exatamente 365 valores por ano, sem 29 de fevereiro. $N = 11680 = 32 \times 365$, de modo que o ciclo anual coincide exatamente com o harmônico $k = 32$ ($11680/32 = 365{,}000$ dias).

In [ ]:
URL_CLIVAC = ("https://clivac.eri.ucsb.edu/wp-content/uploads/"
              "temp_1000mb_lat37N_lon120W_series_only.dat")


ARQUIVO_CLIVAC = "temp_1000mb_lat37N_lon120W_series_only.dat"


def carregar_serie_clivac(arquivo=ARQUIVO_CLIVAC):
    """Série do exercício: colunas year, mon, day, Temp."""
    fonte, alvo = localizar_dado(arquivo, URL_CLIVAC)
    df = pd.read_csv(alvo, sep=r"\s+")
    df.columns = ["ano", "mes", "dia", "temp"]
    data = pd.to_datetime(dict(year=df.ano, month=df.mes, day=df.dia))
    serie = pd.Series(df.temp.astype(float).values, index=data, name="T1000_37N_120W")
    serie.attrs["fonte"] = fonte
    return serie


def serie_clivac_sintetica(n=11680, semente=2):
    """Plano B: ciclo anual de 8 K + ruído vermelho + sinal intrassazonal, em K."""
    rng = np.random.default_rng(semente)
    nn = np.arange(n, dtype=float)
    ruido = np.zeros(n)
    for i in range(1, n):
        ruido[i] = 0.85 * ruido[i - 1] + rng.normal(0, 1)
    vals = (292.0
            + 8.0 * np.cos(2 * np.pi * (nn - 200) / 365)   # ciclo anual
            + 2.0 * ruido / ruido.std()                     # sinótica
            + 0.6 * np.sin(2 * np.pi * nn / 45))            # intrassazonal

    return pd.Series(vals, index=pd.date_range("1979-01-01", periods=n, freq="D"),
                     name="T1000_SINTETICA")


try:
    temp = carregar_serie_clivac()
    print("série do CLIVAC, dado real | fonte:", temp.attrs["fonte"])
except Exception as e:
    print("nenhuma fonte respondeu (%s) -> série sintética" % type(e).__name__)
    temp = serie_clivac_sintetica()

xc = temp.values                       # K
tc = temp.index
mtot = len(xc)
print(temp.name, "| mtot =", mtot, "| de", tc.min().date(), "a", tc.max().date())
print("média %.2f K | desvio-padrão %.2f K | faltantes %d"
      % (np.mean(xc), np.std(xc), int(temp.isna().sum())))
print("dias por ano (valores únicos):", sorted(set(temp.groupby(tc.year).size())))
print("N / 365 =", mtot / 365, "-> o ciclo anual é o harmônico k =", mtot // 365)

Antes de filtrar, a série e o seu espectro.

In [ ]:
fc_temp, Pc_temp = espectro(xc, dt)


def fig_serie_clivac(ano_ini=1979, anos=32, suavizacao=201):
    """A série do exercício e seu espectro."""
    sel = (tc.year >= ano_ini) & (tc.year < ano_ini + anos)
    if sel.sum() < 10:
        print("janela curta demais")
        return
    fig, ax = plt.subplots(2, 1, figsize=(12, 6))
    ax[0].plot(tc[sel], xc[sel], color=CINZA, lw=0.4 if sel.sum() > 3000 else 0.9)
    ax[0].set_ylabel("T em 1000 hPa (K)")
    ax[0].set_title("Série bruta em 37°N, 120°W, %d–%d: ciclo anual predominante"
                    % (tc[sel].year.min(), tc[sel].year.max()))

    ax[1].loglog(1 / fc_temp, suavizar(Pc_temp, suavizacao), color=NAVY, lw=1.2)
    for periodo, rotulo in [(7, "sinótica\n~7 d"), (45, "intrassazonal\n~45 d"),
                            (365, "anual"), (365 * 4, "interanual")]:
        ax[1].axvline(periodo, color=CINZA, ls="--", lw=0.8)
        ax[1].text(periodo, ax[1].get_ylim()[1] * 0.25, rotulo, fontsize=8,
                   ha="center", color=CINZA)
    ax[1].set_xlim(2, 8000)
    ax[1].set_xlabel("período (dias)"); ax[1].set_ylabel("densidade espectral")
    ax[1].set_title("Espectro da série completa: o pico anual excede o restante do "
                    "espectro em uma ordem de grandeza")
    plt.tight_layout(); plt.show()


if WIDGETS:
    interact(fig_serie_clivac,
             ano_ini=sl(1979, 1979, 2010, 1, "ano inicial"),
             anos=sl(32, 1, 32, 1, "anos na tela"),
             suavizacao=sl(201, 1, 601, 20, "suavização do espectro"))
else:
    fig_serie_clivac()

### Programas do CLIVAC em Python

As duas sub-rotinas IDL seguem os mesmos quatro passos

1. **Preparar**: remove média e tendência linear (`mean`, `linfit` no IDL);
2. **Transformar**: $H(f) = \mathcal{F}\{X_t\}$ (`FFT(input,/double)`);
3. **Mascarar**: multiplica $H(f)$ por uma resposta $R(\omega)$ que vale 1 na banda desejada e
   0 fora (`band`, montada com `where`);
4. **Voltar**: $Y_t = \mathcal{F}^{-1}\{H(f) R(\omega)\}$, parte real
   (`FFT(tmp,/inverse,/double)`).

Observação: no passo 3 a máscara trabalha com **período**, e o programa original trunca o período para inteiro. A tradução mantém a truncagem, porque ela altera quais harmônicos entram na banda.

In [ ]:
def frequencias_e_periodos(n, delt=1.0):
    """Eixo de frequências e de períodos dos N harmônicos, como o programa original monta.

    A FFT de N pontos devolve N coeficientes: a primeira metade são frequências
    positivas, a segunda metade as negativas. O período P = 1/f é truncado para
    inteiro — um harmônico de 99,7 d passa a valer 99 d na hora de testar a banda.
    """
    f = np.arange(n, dtype=float)
    n21 = n // 2 + 1                       # primeiro índice de frequência negativa
    if n % 2 == 0:
        f[n21:] = n21 - n + np.arange(n21 - 2)
    else:
        f[n21:] = n21 - n + np.arange(n21 - 1)
    f = f / (n * float(delt))

    per = np.zeros(n)
    per[1:] = 1.0 / f[1:]
    per = np.trunc(per)                    # trunca, não arredonda
    return f, per


f_idl, per_idl = frequencias_e_periodos(mtot, dt)
print("período do harmônico 1 (o mais longo):", per_idl[1], "dias  = N*dt")
print("período do harmônico 32:", 1 / f_idl[32], "dias -> truncado:", per_idl[32])

As duas sub-rotinas, passa-baixa e passa-banda, ambas montando a máscara $R(\omega)$ em
período.

In [ ]:
def filtersub_low(cook, cut, delt=1.0):
    """Passa-baixa por FFT: retém os harmônicos de período >= cut (em dias).

    O teste é "maior ou IGUAL": um harmônico de período exatamente igual a `cut`
    é RETIDO, não removido — é isso que decide o Exercício 1.
    """
    n = len(cook)
    _, per = frequencias_e_periodos(n, delt)

    band = np.zeros(n)
    band[(per >= cut) | (per <= -cut)] = 1.0             # |período| >= cut

    entrada = remover_media_e_tendencia(cook)
    ts_freq = np.fft.fft(entrada)
    cook_mjo = np.real(np.fft.ifft(ts_freq * band))
    return cook_mjo, band


def filtersub_band(cook, cut1, cut2, delt=1.0):
    """Passa-banda por FFT: retém cut1 <= |período| < cut2, como diferença de dois
    passa-baixa (o mesmo raciocínio dos pesos de Lanczos, agora aplicado
    à resposta R e não aos pesos).
    """
    n = len(cook)
    _, per = frequencias_e_periodos(n, delt)

    # o elemento 0 é a média, de período 0: cairia dentro de |per| < cut1 e entraria
    # na máscara errada. Recebendo o período do harmônico 1 (o mais longo), fica fora
    # das duas máscaras e a média é descartada, que é o que se quer.
    tmp_per = per.copy()
    tmp_per[0] = tmp_per[1]

    band_hig = np.zeros(n)
    band_hig[(tmp_per > -cut1) & (tmp_per < cut1)] = 1.0   # rápido demais
    band_low = np.zeros(n)
    band_low[(tmp_per > -cut2) & (tmp_per < cut2)] = 1.0   # tudo abaixo do limite superior
    band = band_low - band_hig                             # sobra a faixa entre os dois

    entrada = remover_media_e_tendencia(cook)
    ts_freq = np.fft.fft(entrada)
    cook_mjo = np.real(np.fft.ifft(ts_freq * band))
    return cook_mjo, band


def periodos_retidos(band, per):
    """Quantos harmônicos a máscara reteve, e entre que períodos."""
    p = np.abs(per)[band == 1]
    return int(band.sum()), p.min(), p.max()

### Exercício 1: Passa-baixa com período de corte de 365 dias

Antes do gráfico, a verificação dos harmônicos retidos pela máscara.

In [ ]:
y_ex1, band_ex1 = filtersub_low(xc, cut=365, delt=dt)

n_ret, p_min, p_max = periodos_retidos(band_ex1, per_idl)
print("harmônicos retidos: %d de %d" % (n_ret, mtot))
print("períodos retidos: de %.0f a %.0f dias" % (p_min, p_max))
print("harmônico anual (k=32, período %.0f d) retido pela máscara? %s"
      % (per_idl[32], "SIM" if band_ex1[32] == 1 else "não"))
print()
print("desvio-padrão: bruta %.2f K -> filtrada %.2f K" % (np.std(xc), np.std(y_ex1)))


def amplitude_anual(serie, periodo=365.0):
    """Amplitude do harmônico anual e fração da variância que ele explica.

    Ajusta y = A0 + A1 cos(2 pi t/P) + B1 sen(2 pi t/P); a amplitude é
    sqrt(A1^2 + B1^2). É a análise harmônica da aula de sazonalidade, usada
    aqui como régua.
    """
    y = np.asarray(serie, dtype=float)
    ok = np.isfinite(y)
    nn = np.arange(len(y), dtype=float)[ok]
    A = np.c_[np.ones(ok.sum()), np.cos(2 * np.pi * nn / periodo),
              np.sin(2 * np.pi * nn / periodo)]
    c = np.linalg.lstsq(A, y[ok], rcond=None)[0]
    ajuste = A @ c
    return np.hypot(c[1], c[2]), np.var(ajuste) / np.var(y[ok])


for rotulo, serie in [("bruta   ", xc), ("filtrada", y_ex1)]:
    amp, frac = amplitude_anual(serie)
    print("%s: ciclo anual com amplitude %.2f K, explica %2.0f%% da variância"
          % (rotulo, amp, 100 * frac))

In [ ]:
def fig_exercicio1(cut=365):
    """Exercício 1: passa-baixa por FFT. Mexa no corte e veja o ciclo anual entrar ou sair."""
    y, band = filtersub_low(xc, cut=cut, delt=dt)
    n_ret, p_min, _ = periodos_retidos(band, per_idl)
    amp, frac = amplitude_anual(y)

    fig, ax = plt.subplots(2, 1, figsize=(12, 6.4))
    ax[0].plot(tc, xc - np.mean(xc), color=CINZA, lw=0.3, label="bruta (anomalia)")
    ax[0].plot(tc, y, color=NAVY, lw=1.4, label="passa-baixa FFT, corte em %d d" % cut)
    ax[0].axhline(0, color="k", lw=0.7)
    ax[0].set_ylabel("anomalia de T (K)"); ax[0].legend()
    ax[0].set_title("(a) %d harmônicos retidos | ciclo anual com %.2f K "
                    "(era 8,08 K na bruta) | desvio-padrão %.2f K"
                    % (n_ret, amp, np.std(y)))

    fd, Pd = espectro(y, dt)
    ax[1].loglog(1 / fc_temp, suavizar(Pc_temp, 9), color=CINZA, lw=0.9, label="bruta")
    ax[1].loglog(1 / fd, suavizar(Pd, 9), color=NAVY, lw=0.9, label="filtrada")
    ax[1].axvline(cut, color=VERM, ls="--", lw=1.2)
    ax[1].text(cut, ax[1].get_ylim()[1] * 0.2, " corte", color=VERM, fontsize=9)
    ax[1].set_xlim(2, 8000)
    ax[1].set_xlabel("período (dias)"); ax[1].set_ylabel("densidade espectral"); ax[1].legend()
    ax[1].set_title("(b) potência restrita a períodos >= %d d; menor período retido: %.0f d"
                    % (cut, p_min))
    plt.tight_layout(); plt.show()


if WIDGETS:
    interact(fig_exercicio1, cut=sl(365, 30, 2000, 1, "corte (dias)"))
else:
    fig_exercicio1()

In [ ]:
# mesmo filtro, mudando só o corte: 365 -> 366 -> 730 dias
for cut in [365, 366, 730]:
    y, band = filtersub_low(xc, cut=cut, delt=dt)
    n_ret, p_min, _ = periodos_retidos(band, per_idl)
    amp, _ = amplitude_anual(y)
    print("cut = %3d d | harmônicos retidos %3d | menor período retido %4.0f d "
          "| ciclo anual %.2f K | desvio-padrão %.2f K"
          % (cut, n_ret, p_min, amp, np.std(y)))

def fig_compara_cortes(cut_a=365, cut_b=730):
    """Dois cortes, lado a lado, na mesma série e no mesmo programa."""
    ya = filtersub_low(xc, cut=cut_a, delt=dt)[0]
    yb = filtersub_low(xc, cut=cut_b, delt=dt)[0]
    amp_a, _ = amplitude_anual(ya)
    amp_b, _ = amplitude_anual(yb)

    fig, ax = plt.subplots(figsize=(12, 3.6))
    ax.plot(tc, ya, color=VERM, lw=1.2,
            label="cut = %d d — ciclo anual %.2f K" % (cut_a, amp_a))
    ax.plot(tc, yb, color=NAVY, lw=1.8,
            label="cut = %d d — ciclo anual %.2f K" % (cut_b, amp_b))
    ax.axhline(0, color="k", lw=0.7)
    ax.set_ylabel("anomalia (K)"); ax.legend()
    ax.set_title("Mesma série e mesmo filtro: efeito do período de corte sobre o "
                 "resultado")
    plt.show()


if WIDGETS:
    interact(fig_compara_cortes,
             cut_a=sl(365, 30, 2000, 1, "corte A (dias)"),
             cut_b=sl(730, 30, 2000, 1, "corte B (dias)"))
else:
    fig_compara_cortes()

### Exercício 2: Passa-banda com cortes em 20 e 100 dias

Passa-banda com cortes em 20 e 100 dias. Novamente, a verificação da máscara em primeiro lugar.

In [ ]:
cut1, cut2 = 20, 100
y_ex2, band_ex2 = filtersub_band(xc, cut1, cut2, delt=dt)

n_ret2, p_min2, p_max2 = periodos_retidos(band_ex2, per_idl)
print("harmônicos retidos: %d de %d" % (n_ret2, mtot))
print("períodos retidos: de %.0f a %.0f dias" % (p_min2, p_max2))
print("  <- o limite superior é 99, não 100: o período foi truncado para inteiro,")
print("     de modo que o harmônico de 99,7 d conta como 99 e ainda entra na banda.")
print()
print("desvio-padrão: bruta %.2f K -> banda 20–100 d %.2f K (%.1f%% da variância)"
      % (np.std(xc), np.std(y_ex2), 100 * np.var(y_ex2) / np.var(xc)))
amp_ex2, _ = amplitude_anual(y_ex2)
print("ciclo anual residual: %.4f K (era 8,08 K na bruta)" % amp_ex2)
print("média da série filtrada: %.2e K (a máscara zerou o harmônico 0)" % y_ex2.mean())

In [ ]:
janela = slice(7300, 8400)          # ~3 anos no meio da série
fd_ex2, Pd_ex2 = espectro(y_ex2, dt)


def fig_exercicio2(corte1=20, corte2=100, ano_ini=1999, anos=3):
    """Exercício 2: passa-banda por FFT, com os cortes e a janela à sua escolha."""
    if corte1 >= corte2:
        print("o corte 1 tem de ser menor que o corte 2")
        return
    y, band = filtersub_band(xc, corte1, corte2, delt=dt)
    n_ret, p_min, p_max = periodos_retidos(band, per_idl)
    sel = (tc.year >= ano_ini) & (tc.year < ano_ini + anos)

    fig, ax = plt.subplots(2, 1, figsize=(12, 6.4))
    ax[0].plot(tc[sel], xc[sel] - np.mean(xc), color=CINZA, lw=0.7, label="bruta (anomalia)")
    ax[0].plot(tc[sel], y[sel], color=VERDE, lw=1.8,
               label="passa-banda %d–%d d" % (corte1, corte2))
    ax[0].axhline(0, color="k", lw=0.7); ax[0].set_ylabel("anomalia (K)"); ax[0].legend()
    ax[0].set_title("(a) %d harmônicos retidos, %.1f%% da variância, desvio-padrão %.2f K"
                    % (n_ret, 100 * np.var(y) / np.var(xc), np.std(y)))

    fd, Pd = espectro(y, dt)
    ax[1].loglog(1 / fc_temp, suavizar(Pc_temp, 9), color=CINZA, lw=0.9, label="bruta")
    ax[1].loglog(1 / fd, suavizar(Pd, 9), color=VERDE, lw=0.9, label="filtrada")
    ax[1].axvspan(corte1, corte2, color=VERDE, alpha=0.12)
    ax[1].set_xlim(2, 8000)
    ax[1].set_xlabel("período (dias)"); ax[1].set_ylabel("densidade espectral"); ax[1].legend()
    ax[1].set_title("(b) variância confinada à faixa %.0f–%.0f d, com queda abrupta "
                    "nas duas bordas" % (p_min, p_max))
    plt.tight_layout(); plt.show()


if WIDGETS:
    interact(fig_exercicio2,
             corte1=sl(20, 2, 300, 1, "corte 1 (dias)"),
             corte2=sl(100, 5, 1000, 5, "corte 2 (dias)"),
             ano_ini=sl(1999, 1979, 2010, 1, "ano inicial"),
             anos=sl(3, 1, 32, 1, "anos na tela"))
else:
    fig_exercicio2()

In [ ]:
media = np.mean(xc)

def fig_filtrada_sobre_bruta(ano_ini=1979, anos=2):
    """A banda intrassazonal com a média somada de volta, sobre a série bruta."""
    sel = (tc.year >= ano_ini) & (tc.year < ano_ini + anos)
    if sel.sum() == 0:
        print("janela vazia")
        return
    fig, ax = plt.subplots(figsize=(12, 3.6))
    ax.plot(tc[sel], xc[sel], color=CINZA, lw=0.9, label="série bruta")
    ax.plot(tc[sel], y_ex2[sel] + media, color=VERDE, lw=2.2,
            label="filtrada, com a média somada de volta")
    ax.set_ylabel("T em 1000 hPa (K)"); ax.legend()
    ax.set_title("%d–%d: banda intrassazonal sobreposta à série bruta"
                 % (tc[sel].year.min(), tc[sel].year.max()))
    plt.show()


if WIDGETS:
    interact(fig_filtrada_sobre_bruta,
             ano_ini=sl(1979, 1979, 2010, 1, "ano inicial"),
             anos=sl(2, 1, 32, 1, "anos na tela"))
else:
    fig_filtrada_sobre_bruta()